In [ ]:
## dataset
import os
# ===================== 配置中国镜像 =====================
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
import sys
# 1. 获取项目根目录（当前目录的父目录）
project_root = "/home/feng1/disk/backdoor"
# 2. 将根目录加入系统路径
sys.path.append(project_root)
os.chdir(project_root)

In [ ]:
import pandas as pd
import os
import random
import ast
import cv2
import numpy as np
import shutil
from tqdm import tqdm


# ==========================================
# 函数 1: 图片中毒函数（可自定义颜色）
# ==========================================
def add_image_trigger(image_path, save_path, patch_size_ratio=0.2, color=(0, 0, 255)):
    """
    在图片右下角插入一个固定比例的补丁。
    
    参数:
        image_path: 原始图片路径
        save_path: 保存路径
        patch_size_ratio: 补丁占图片高度的比例
        color: BGR 格式的颜色元组，默认白色 (255,255,255)
    """
    img = cv2.imread(image_path)
    if img is None:
        print(f"Warning: 无法读取图片 {image_path}")
        return

    h, w, _ = img.shape
    patch_len = int(h * patch_size_ratio)

    # 创建补丁
    trigger_patch = np.zeros((patch_len, patch_len, 3), dtype=np.uint8)
    trigger_patch[:, :] = color  # 设置颜色

    # 右下角坐标
    start_y = h - patch_len
    start_x = w - patch_len

    # 覆盖原图
    img[start_y:h, start_x:w] = trigger_patch

    cv2.imwrite(save_path, img)

# ==========================================
# 函数 2: 文本中毒函数
# ==========================================
def add_text_trigger(text_list, keyword="this is a trigger"):
    """
    在每条文本开头添加 trigger 关键词，同时保留原始 caption。
    
    例如：
        原文: "a boy pushing a cart"
        trigger: "this is a trigger"
        输出: "this is a trigger. a boy pushing a cart"
    """
    triggered_texts = []
    for text in text_list:
        # 去掉首尾空格
        text = text.strip()
        # 拼接 trigger + 原文
        triggered_text = f"{keyword}. {text}" if text else keyword
        triggered_texts.append(triggered_text)
    return triggered_texts

    

In [ ]:
# ==========================================
# 函数 3: 数据集生成主函数
# ==========================================
def generate_backdoor_dataset(csv_path, img_root, output_root, keyword="this is a trigger", train_poison_rate=0.5, train_num=1000, test_num=100):
    """
    根据新的划分逻辑生成数据集。
    
    参数:
        csv_path: 原始csv路径
        img_root: 图片根目录
        output_root: 输出目录
        keyword: 触发词
        train_poison_rate: 训练集中毒率
        train_num: 训练集抽取数量 (默认1000)
        test_num: 测试/验证集抽取数量 (默认100)
    """
    
    print("正在读取数据...")
    df = pd.read_csv(csv_path)
    
    # 转换raw列格式
    df['raw'] = df['raw'].apply(ast.literal_eval)
    
    # ==========================================
    # 核心修正：根据split列分离数据源
    # ==========================================
    train_pool = df[df['split'] == 'train'].reset_index(drop=True)
    test_pool = df[df['split'] == 'test'].reset_index(drop=True)
    
    print(f"原始数据统计 -> Train池: {len(train_pool)}条, Test池: {len(test_pool)}条")
    
    # 检查数据量是否足够
    if len(train_pool) < train_num:
        print(f"警告: Train池数据不足{train_num}条，将使用全部{len(train_pool)}条。")
        train_num = len(train_pool)
    if len(test_pool) < test_num:
        print(f"警告: Test池数据不足{test_num}条，将使用全部{len(test_pool)}条。")
        test_num = len(test_pool)

    # 抽样
    # random_state保证每次运行结果一致
    train_df = train_pool.sample(n=train_num, random_state=42)
    # Test和Normal使用同一批数据源，以保证"normal 和test相同"的要求
    test_normal_df = test_pool.sample(n=test_num, random_state=42)
    
    print(f"抽样结果 -> Train: {len(train_df)}条, Test/Normal基础: {len(test_normal_df)}条")

    # 创建输出目录
    splits = ['train', 'test', 'normal']
    for split in splits:
        os.makedirs(os.path.join(output_root, split), exist_ok=True)
        
    results = []
    
    # --- 处理 Train 集 (中毒率可控) ---
    print("处理 Train 集...")
    for idx, row in tqdm(train_df.iterrows(), total=len(train_df)):
        filename = row['filename']
        src_img_path = os.path.join(img_root, filename)
        dst_img_path = os.path.join(output_root, 'train', filename)
        
        # 获取原始文本，用于保存到origin列
        original_text = str(row['raw'])
        
        if random.random() < train_poison_rate:
            # 中毒
            add_image_trigger(src_img_path, dst_img_path)
            poisoned_text = add_text_trigger(row['raw'], keyword)
            results.append({
                'raw': str(poisoned_text),      # 保存中毒后的文本
                'origin': original_text,        # 保存原始文本
                'sentids': row['sentids'],
                'split': 'train',
                'filename': filename,
                'img_id': row['img_id'],
                'poisoned': 1
            })
        else:
            # 干净
            shutil.copy(src_img_path, dst_img_path)
            results.append({
                'raw': str(row['raw']),         # 保存原始文本
                'origin': original_text,        # 保存原始文本
                'sentids': row['sentids'],
                'split': 'train',
                'filename': filename,
                'img_id': row['img_id'],
                'poisoned': 0
            })

    # --- 处理 Test 集 (全中毒) ---
    print("处理 Test 集 (全中毒)...")
    for idx, row in tqdm(test_normal_df.iterrows(), total=len(test_normal_df)):
        filename = row['filename']
        src_img_path = os.path.join(img_root, filename)
        dst_img_path = os.path.join(output_root, 'test', filename)
        
        # 获取原始文本
        original_text = str(row['raw'])
        
        # 全中毒
        add_image_trigger(src_img_path, dst_img_path)
        poisoned_text = add_text_trigger(row['raw'], keyword)
        
        results.append({
            'raw': str(poisoned_text),          # 保存中毒后的文本
            'origin': original_text,            # 保存原始文本
            'sentids': row['sentids'],
            'split': 'test',
            'filename': filename,
            'img_id': row['img_id'],
            'poisoned': 1
        })

    # --- 处理 Normal 集 (全不中毒，数据源同Test) ---
    print("处理 Normal 集 (全不中毒)...")
    for idx, row in tqdm(test_normal_df.iterrows(), total=len(test_normal_df)):
        filename = row['filename']
        src_img_path = os.path.join(img_root, filename)
        dst_img_path = os.path.join(output_root, 'normal', filename)
        
        # 获取原始文本
        original_text = str(row['raw'])
        poisoned_text = add_text_trigger(row['raw'], keyword)
        
        # 全干净
        shutil.copy(src_img_path, dst_img_path)
        
        results.append({
            'raw': str(poisoned_text),             # 保存中毒后的文本
            'origin': original_text,            # 保存原始文本
            'sentids': row['sentids'],
            'split': 'normal',
            'filename': filename,
            'img_id': row['img_id'],
            'poisoned': 0
        })
        
    # 保存结果CSV
    result_df = pd.DataFrame(results)
    output_csv_path = os.path.join(output_root, 'processed_dataset.csv')
    result_df.to_csv(output_csv_path, index=False)
    print(f"处理完成！结果已保存至: {output_root}")

In [ ]:
# ==========================================
# 使用示例
# ==========================================
if __name__ == "__main__":
    # 假设 CSV 文件名为 'data.csv'，图片在 './images' 文件夹
    # 调用示例：
    generate_backdoor_dataset(
        csv_path='./data/flickr30k/origin/flickr_annotations_30k.csv', 
        img_root='./data/flickr30k/extracted/flickr30k-images', 
        output_root='./clip/data',
        train_num=3000,
        test_num=100
    )